In [1]:
from dotenv import load_dotenv

load_dotenv()


True

# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. In particular: 1) added error handling for MCP failures, 2) limited the number of searches. It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

## Setup Tools


In [2]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [3]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

/tmp/ipykernel_685129/2549610831.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Create State

In [26]:
from langchain.agents import AgentState

# define WeddingState(AgentState) with string fields:
# origin, destination, guest_count, genre
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: int
    genre: str

## Create Subagents


In [10]:
from langchain.agents import create_agent

# create `travel_agent` with create_agent
# - model="gpt-5-nano", tools=tools (the MCP flight tools)
# - system_prompt: act as a travel agent, no follow-up questions, find one cheap/short
#   one-way economy flight at a good time of year, retry if the MCP tool fails,
#   then return a shortlist
travel_agent = create_agent(
    model="gpt-5-nano", 
    tools=tools,
    system_prompt="act as a travel agent, no follow-up questions, find one cheap/short one-way economy flight at a good time of year, retry if the MCP tool fails, then return a shortlist"
)

In [12]:
# create `venue_agent`
# - tools=[web_search]
# - system_prompt: venue specialist; pick by price (lowest), capacity (exact match),
#   reviews (highest); no follow-ups; stop after 12 web searches and summarize
venue_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt="system_prompt: venue specialist; pick by price (lowest), capacity (exact match), reviews (highest); no follow-ups; stop after 12 web searches and summarize"
)

In [15]:
# create `playlist_agent`
# - tools=[query_playlist_db]
# - system_prompt: playlist specialist; discover the SQLite schema first, fix failed
#   queries, keep trying, then report total duration and cost of the playlist
playlist_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt="playlist specialist; discover the SQLite schema first, fix failed queries, keep trying, then report total duration and cost of the playlist"
)

## Main Coordinator


In [35]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

# write four @tool functions that take `runtime: ToolRuntime`:
# - search_flights (async): read origin/destination from runtime.state, ainvoke travel_agent
# - search_venues: read destination/guest_count, invoke venue_agent
# - suggest_playlist: read genre, invoke playlist_agent
#   (each returns response['messages'][-1].content)
# - update_state(origin, destination, guest_count, genre, runtime): return a Command
#   that updates state AND includes a ToolMessage with runtime.tool_call_id
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """The travel agent will search flights from origin to destination"""
    try:
        origin, destination = runtime.state["origin"], runtime.state['destination']
        response = await travel_agent.ainvoke(
            { "messages": [HumanMessage(content=f"Search for flights with origin {origin} and destination {destination}")]},
        )
        return response['messages'][-1].content
    except KeyError:
        return "No origin or destination found in state"

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """The venue agent will perform a web search to find the best venue based on various categories"""
    try:
        destination, guest_count = runtime.state['destination'], runtime.state["guest_count"]
        response = venue_agent.invoke(
            { "messages": [HumanMessage(content=f"Search for venues with location {destination} and can fit {guest_count} guests")]},
        )
        return response['messages'][-1].content
    except KeyError:
        return "No destination or guest_count found in state"

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """The playlist agent will find tracks for a specific genre"""
    try:
        genre = runtime.state['genre']
        response = playlist_agent.invoke(
            { "messages": [HumanMessage(content=f"Find tracks with genre {genre} for a wedding playlist")]},
        )
        return response['messages'][-1].content
    except KeyError:
        return "No genre found in state"

@tool
def update_state(origin: str, destination: str, guest_count: int, genre: str, runtime: ToolRuntime) -> Command:
    """Updating agent state based off if you know all of origin, destination, guest_count, genre"""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated origin, destination, guest_count, genre", tool_call_id=runtime.tool_call_id)]}
    )

In [36]:
from langchain.agents import create_agent

# create `coordinator` with the four tools above,
# state_schema=WeddingState, and a system_prompt telling it to first gather info and
# update state, then delegate to the specialists, then put together the wedding plan
coordinator = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="You are a wedding planner. " \
    "I would like you to gather all the info you can and update state then delegate searching flights, searching venues, suggesting playlists to the respective specialist agents. " \
    "Once you have gotten all the information from the specialist agents, please put together the wedding plan. "
)

## Test


In [37]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [38]:
from pprint import pprint

pprint(response)

{'destination': 'Paris',
 'genre': 'jazz',
 'guest_count': 100,
 'messages': [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='5c1b906b-d117-4493-9f76-647f533d0314'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 293, 'prompt_tokens': 293, 'total_tokens': 586, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EY35GHVK3ZblT7WLbY79UeKYDhwie', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12ea1-b7b4-7d53-b86d-e0f1df3ef550-0', tool_calls=[{'name': 'update_state', 'args': {'or

In [39]:
print(response["messages"][-1].content)

Here’s the initial wedding plan collated from flights, venues, and a jazz playlist brief. I’ve organized it so you can quickly review options and decide what to pursue next. I’ve also included clear next steps and questions to refine.

State recap
- Origin: London
- Destination: Paris
- Guest count: 100
- Music genre: Jazz (for the vibe of the event)

1) Flights (to help with guest arrivals and travel planning)
Notes:
- Date in the flight results: 20 Nov 2026 (please confirm the actual event date; I’ll adjust flight searches to match your date once confirmed)
- Several one-way economy nonstop options from various London airports to Paris airports were identified

Top flight options to consider (summary)
- SEN → CDG (London Southend to Paris Charles de Gaulle): 14:25–16:35, 57 EUR, nonstop (best price)
  - Pros: Cheapest, nonstop
  - Cons: London Southend is less central than Heathrow/Gatwick
- LHR → ORY (London Heathrow to Paris Orly): 15:15–17:45, 63 EUR, nonstop
  - Pros: Main intern

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r